# NFHS-5 Stunting — clean pipeline
Simplified from the original 106-cell notebook. Produces the **exact same**
`final_table`, `map_data`, `state_stunting`, and `stunting_map` objects —
just without the exploration/debugging cells.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

KR_PATH    = r"C:\Users\91981\Downloads\IAKR7EDT\IAKR7EFL.DTA"
INDIA_JSON = r"C:\Users\91981\Desktop\Stunting folder\india.json"

## 1. Load & clean stunting

In [ ]:
df = pd.read_stata(KR_PATH, convert_categoricals=False)

# height-for-age z-score: blank out special missing codes (>=9996)
df["hw70_clean"] = df["hw70"].where(df["hw70"] < 9996, np.nan)

# plausibility guard: WHO flags z-scores beyond +/-6 SD as implausible
# (data/measurement errors, not real children). Stored x100, so +/-600.
n_before = df["hw70_clean"].notna().sum()
df["hw70_clean"] = df["hw70_clean"].where(df["hw70_clean"].between(-600, 600), np.nan)
n_dropped = n_before - df["hw70_clean"].notna().sum()
print(f"plausibility guard dropped {n_dropped:,} implausible z-scores "
      f"({100*n_dropped/n_before:.2f}% of measured)")

# stunted = HAZ below -2 SD (stored x100, so < -200); survey weight
df["stunted"] = (df["hw70_clean"] < -200).astype(int)
df["weight"]  = df["v005"] / 1_000_000

# children with a valid measurement (under-5s with plausible anthropometry)
valid = df[df["hw70_clean"].notna()].copy()

national = np.average(valid["stunted"], weights=valid["weight"])
print(f"Rows: {df.shape[0]:,} | valid: {len(valid):,} | national stunting: {100*national:.1f}%")

## 2. State code → name

In [ ]:
state_map = {
 1:'jammu & kashmir',2:'himachal pradesh',3:'punjab',4:'chandigarh',5:'uttarakhand',
 6:'haryana',7:'nct of delhi',8:'rajasthan',9:'uttar pradesh',10:'bihar',11:'sikkim',
 12:'arunachal pradesh',13:'nagaland',14:'manipur',15:'mizoram',16:'tripura',
 17:'meghalaya',18:'assam',19:'west bengal',20:'jharkhand',21:'odisha',22:'chhattisgarh',
 23:'madhya pradesh',24:'gujarat',25:'dadra & nagar haveli and daman & diu',
 27:'maharashtra',28:'andhra pradesh',29:'karnataka',30:'goa',31:'lakshadweep',
 32:'kerala',33:'tamil nadu',34:'puducherry',35:'andaman & nicobar islands',
 36:'telangana',37:'ladakh'
}

## 3. Weighted per-state rates
One helper does the weighted mean; one loop builds every state's poor rate,
rich rate, gap, and overall stunting in a single pass.

In [ ]:
def wmean(d):
    return np.average(d["stunted"], weights=d["weight"]) if len(d) else np.nan

rows = []
for code_ in sorted(valid["v024"].unique()):
    s    = valid[valid["v024"] == code_]
    poor = s[s["v190"] == 1]
    rich = s[s["v190"] == 5]
    if len(poor) == 0 or len(rich) == 0:
        continue
    pr, rr = wmean(poor), wmean(rich)
    rows.append({
        "state_code": code_,
        "poor_rate":  pr,
        "rich_rate":  rr,
        "gap":        pr - rr,
        "stunting_rate": wmean(s),
    })

state_gap = pd.DataFrame(rows)
state_gap["state"]        = state_gap["state_code"].map(state_map)
state_gap["gap_pp"]       = (state_gap["gap"] * 100).round(1)
state_gap["poor_pct"]     = (state_gap["poor_rate"] * 100).round(1)
state_gap["rich_pct"]     = (state_gap["rich_rate"] * 100).round(1)
state_gap["stunting_pct"] = (state_gap["stunting_rate"] * 100).round(1)

# wealth-gap table (same columns/order as the original final_table)
final_table = (state_gap[["state","poor_rate","rich_rate","gap_pp","poor_pct","rich_pct"]]
               .sort_values("gap_pp", ascending=False)
               .reset_index(drop=True))

# overall-stunting table (same as the original state_stunting)
state_stunting = state_gap[["v024" if False else "state_code","stunting_rate","stunting_pct","state"]].copy()
state_stunting = state_stunting.rename(columns={"state_code":"v024"})

final_table.head(10)

## 4. Save the gap table

In [ ]:
final_table.to_csv("state_stunting_gap.csv", index=False)
print("saved state_stunting_gap.csv")

## 5. Merge onto geometry
Loads india.json, normalises names, applies the name crosswalk, and merges.
Produces both `map_data` (wealth gap) and `stunting_map` (overall stunting).

In [ ]:
states = gpd.read_file(INDIA_JSON)
states["name_clean"] = states["name"].str.lower().str.strip()
# fix the one accented name so it matches
states["name_clean"] = states["name_clean"].replace({
    "dādra and nagar haveli and damān and diu": "dadra & nagar haveli and daman & diu"
})

# crosswalk: NFHS naming -> geojson naming
NAME_FIX = {
    "andaman & nicobar islands": "andaman and nicobar",
    "nct of delhi":              "delhi",
    "jammu & kashmir":           "jammu and kashmir",
    "odisha":                    "orissa",
    "uttarakhand":               "uttaranchal",
}
def to_geo_name(s):
    return s.str.lower().str.strip().replace(NAME_FIX)

final_table["state_clean"]    = to_geo_name(final_table["state"])
state_stunting["state_clean"] = to_geo_name(state_stunting["state"])

map_data     = states.merge(final_table,    left_on="name_clean", right_on="state_clean", how="left")
stunting_map = states.merge(state_stunting, left_on="name_clean", right_on="state_clean", how="left")

print("gap unmatched:",      map_data["gap_pp"].isna().sum())
print("stunting unmatched:", stunting_map["stunting_pct"].isna().sum())

## 6. Maps

In [ ]:
import matplotlib.pyplot as plt

def quick_map(gdf, column, title, sub):
    fig, ax = plt.subplots(figsize=(12,12))
    fig.patch.set_facecolor("#f4f1ea"); ax.set_facecolor("#f4f1ea")
    gdf.plot(column=column, cmap="Reds", linewidth=0.6, edgecolor="white",
             legend=True, ax=ax,
             missing_kwds={"color":"#d9d9d9","label":"No data"})
    ax.set_title(title, fontsize=22, fontweight="bold", pad=20)
    ax.axis("off")
    plt.figtext(0.5, 0.06, sub, ha="center", fontsize=9, color="dimgray")
    plt.show()

quick_map(stunting_map, "stunting_pct",
          "The Burden of Child Stunting Across India\nNFHS-5",
          "Percentage of children under five who are stunted. Source: NFHS-5 microdata.")

quick_map(map_data, "gap_pp",
          "India's Nutrition Inequality Divide\nNFHS-5",
          "Poorest-minus-richest stunting gap (percentage points). Source: NFHS-5 microdata.")